In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_15_markov_chains_and_metropolis.yaml')
ws.start()


$\def\AA{\text{Å}}$

# MSE 485 · Lesson 15: Markov Chains, Detailed Balance & Metropolis Monte Carlo

_Session 8cc10961 · Estimated time: ~115 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Explain why direct importance sampling fails exponentially as exp(-aN) in high-dimensional configuration space and how Markov Chain Monte Carlo (MCMC) overcomes this barrier.
- Formulate a discrete-time Markov chain via a stochastic transition matrix P(s -> s') and evolve state distributions using the master equation f_{n+1} = f_n P.
- Connect the eigenvalue spectrum of P to stationary equilibrium (eps_0 = 1), geometric convergence (|eps_1|^n), equilibration time tau_eq, and the three failure modes of ergodicity.
- Solve the 2-state computer reliability Markov chain analytically and iteratively for its transient evolution, stationary distribution pi(infinity), and spectral gap.
- Derive the Metropolis-Hastings acceptance criterion from detailed balance, compare it with Barker/Heat-Bath acceptance via Peskun's theorem, and explain why the partition function Z cancels.
- Analyze single-particle canonical trial moves, diagnose the two classic MCMC implementation bugs, tune step size Delta for optimal phase-space diffusion, and contrast MC with MD.


In [2]:
ws.render_section('sec_markov_master')

## 1. Why Direct Sampling Fails & The Markov Chain Master Equation

In Lesson 14 we saw that Monte Carlo integration estimates high-dimensional expectation values by drawing configurations from a probability distribution. However, in statistical mechanics we almost never know how to sample the canonical Boltzmann distribution $\pi(s) = \frac{1}{Z}e^{-\beta E(s)}$ directly when $3N \gg 1$.

### The Exponential Overlap Catastrophe in Direct Sampling
Suppose we attempt **direct importance sampling** by drawing configurations independently from a simpler reference distribution $g(s)$ (such as a uniform distribution or an ideal gas) and reweighting by $\pi(s)/g(s)$. Because configuration space has $3N$ degrees of freedom, the overlap between any tractable reference distribution $g(s)$ and the true interacting Boltzmann distribution $\pi(s)$ decays **exponentially with system size**:
$$\text{Overlap}(\pi, g) \sim \exp(-a N)$$
where $a > 0$ is an intensive constant related to the excess free energy per particle. Consequently, almost every randomly generated configuration lands in a region of core overlap where $e^{-\beta E(s)} \approx 0$, and the statistical error bars grow **exponentially** as $\sim e^{+aN/2}$.

### Markov Chains: Memoryless Random Walks Through Phase Space
Introduced by **Metropolis, Rosenbluth, Rosenbluth, Teller, and Teller (1953)** on the Los Alamos MANIAC computer to calculate the equation of state of a 2D hard-sphere liquid, **Markov Chain Monte Carlo (MCMC)** replaces independent guessing with an importance-weighted **random walk** through configuration space:
$$s_1 \to s_2 \to s_3 \to \cdots \to s_n \to s_{n+1} \to \cdots$$
Here $s$ denotes a complete microstate of the system (e.g., all $3N$ particle coordinates $\mathbf{R} = \{\mathbf{r}_1, \dots, \mathbf{r}_N\}$ or all $N$ lattice spins).

By definition, a discrete-time **Markov chain** is governed by a time-independent conditional **transition probability** $P(s \to s')$: the probability distribution of the next state $s_{n+1}$ depends **only** on the current state $s_n$ and not on any earlier history $s_1, \dots, s_{n-1}$ (*"a drunkard has no memory!"*).

### Properties of the Stochastic Transition Matrix $P$
The matrix $P$ with entries $P_{s,s'} = P(s \to s')$ is a **right stochastic matrix**, meaning it must satisfy two fundamental axioms:
1. **Non-negativity:** Every transition probability is real and non-negative:
   $$P(s \to s') \ge 0 \quad \forall\, s, s'$$
2. **Row Normalization (Conservation of Probability):** From any starting state $s$, the system must transition *somewhere* (including possibly staying in the same state $s' = s$) with total probability $1$:
   $$\sum_{s'} P(s \to s') = 1 \quad \forall\, s$$

### The Discrete-Time Master Equation
Let $f_n(s)$ denote the probability that the system occupies microstate $s$ after $n$ steps of the random walk (represented as a **row vector** $f_n$). After one additional step, the probability distribution $f_{n+1}(s')$ is obtained by summing over all possible predecessor states $s$:
$$f_{n+1}(s') = \sum_{s} f_n(s)\, P(s \to s') \qquad \Longleftrightarrow \qquad f_{n+1} = f_n P$$
Iterating this **master equation** $n$ times from an initial distribution $f_0$ yields:
$$f_n = f_0 P^n$$
where $P^n$ is the $n$-step transition matrix. Stationary (invariant) states of the random walk are **left eigenvectors** $\phi$ of $P$:
$$\phi P = \varepsilon \phi$$
with eigenvalue $\varepsilon = 1$. Notice simultaneously that the column vector of ones $\mathbf{1} = (1, 1, \dots, 1)^T$ is always a **right eigenvector** of $P$ with eigenvalue $1$ because $\sum_{s'} P(s \to s')\cdot 1 = 1$.


In [3]:
ws.render_section('q15_01_direct_sampling_overlap')

**Q1 · easy** · 💡

Why does direct importance sampling from a non-interacting reference distribution $g(s)$ fail for a dense fluid of $N \sim 10^3$ interacting atoms?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the probability overlap between two different many-body distributions scales with the number of particles $N$.

</div>
</details>

- [ ] **(A)** The overlap between the reference distribution and the true Boltzmann distribution decays exponentially as $\exp(-aN)$, causing variance and error bars to blow up exponentially with $N$.
- [ ] **(B)** Random number generators cannot produce more than $3N$ independent uniform variates without repeating their period.
- [ ] **(C)** Direct sampling violates Liouville's theorem because it does not integrate Newton's equations of motion.
- [ ] **(D)** The normalization constant of a uniform reference distribution $g(s) = V^{-N}$ cannot be computed analytically.

In [4]:
ws.render_section('q15_02_markov_memoryless_property')

**Q2 · easy** · 💡

Which mathematical statement defines the **Markov property** (*'a drunkard has no memory'*) for a sequence of states $s_1 \to s_2 \to \cdots \to s_n \to s_{n+1}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Condition the probability of $s_{n+1}$ on the entire history $s_1, s_2, \dots, s_n$.

</div>
</details>

- [ ] **(A)** $\mathbb{P}(s_{n+1} \mid s_n, s_{n-1}, \dots, s_1) = \mathbb{P}(s_{n+1} \mid s_n) = P(s_n \to s_{n+1})$
- [ ] **(B)** $\mathbb{P}(s_{n+1} \mid s_n) = \mathbb{P}(s_{n+1})$ independent of $s_n$
- [ ] **(C)** $P(s \to s') = P(s' \to s)$ for all pairs of states $s, s'$
- [ ] **(D)** $s_{n+1}$ is never allowed to equal $s_n$ or $s_{n-1}$

In [5]:
ws.render_section('q15_03_stochastic_matrix_normalization')

**Q3 · easy** · 💡

In the convention where $f_n$ is a row vector evolving via $f_{n+1}(s') = \sum_s f_n(s) P(s \to s')$, which normalization condition must every valid transition matrix $P(s \to s')$ satisfy?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Sum over all possible destination states $s'$ starting from a fixed current state $s$.

</div>
</details>

- [ ] **(A)** $P(s \to s') \ge 0$ and $\sum_{s'} P(s \to s') = 1$ for every initial state $s$ (row normalization).
- [ ] **(B)** $P(s \to s') \ge 0$ and $\sum_{s} P(s \to s') = 1$ for every final state $s'$ (column normalization).
- [ ] **(C)** $\sum_{s, s'} P(s \to s') = 1$ across the entire matrix.
- [ ] **(D)** $\sum_{s} [P(s \to s')]^2 = 1$ (unitary normalization).

In [6]:
ws.render_section('q15_04_probability_conservation_proof')

**Q4 · medium** · 💡

Suppose $\sum_s f_n(s) = 1$. When we sum the master equation $f_{n+1}(s') = \sum_s f_n(s) P(s \to s')$ over all destination states $s'$, why is total probability $\sum_{s'} f_{n+1}(s') = 1$ automatically conserved at step $n+1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Interchange the order of summation over $s$ and $s'$.

</div>
</details>

- [ ] **(A)** Because $\sum_{s'} f_{n+1}(s') = \sum_s f_n(s) \left[\sum_{s'} P(s \to s')\right] = \sum_s f_n(s) \cdot 1 = 1$.
- [ ] **(B)** Because $P(s \to s')$ is always a symmetric matrix ($P = P^T$).
- [ ] **(C)** Because the trace of $P$ equals $1$ ($\operatorname{Tr}(P) = 1$).
- [ ] **(D)** Because $\det(P) = 1$ for every stochastic matrix.

In [7]:
ws.render_section('q15_05_missing_matrix_entry')

**Q5 · easy** · 💡

A 3-state Markov chain has the transition matrix:
$$P = \begin{pmatrix} 0.50 & 0.35 & p_{13} \\ 0.20 & 0.60 & 0.20 \\ 0.10 & 0.40 & 0.50 \end{pmatrix}$$
What must the value of $p_{13} = P(1 \to 3)$ be for $P$ to be a valid stochastic matrix?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply row normalization $\sum_{s'=1}^3 P(1 \to s') = 1$ to the first row.

</div>
</details>

_Numeric answer required._

In [8]:
ws.render_section('q15_06_left_vs_right_eigenvector')

**Q6 · medium** · 💡

For a general asymmetric stochastic matrix $P(s \to s')$, how do the **left** and **right** eigenvectors corresponding to the unit eigenvalue $\varepsilon_0 = 1$ differ physically?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $\pi P = \pi$ (row vector) with $P \mathbf{1} = \mathbf{1}$ (column vector).

</div>
</details>

- [ ] **(A)** The left eigenvector $\pi P = \pi$ gives the stationary probability distribution over states, whereas the right eigenvector $P \mathbf{1} = \mathbf{1}$ is trivially the uniform column vector of ones expressing row normalization.
- [ ] **(B)** The right eigenvector gives the stationary distribution, whereas the left eigenvector is always uniform.
- [ ] **(C)** The left and right eigenvectors of $P$ are always identical because every stochastic matrix is Hermitian.
- [ ] **(D)** The right eigenvector gives the relaxation rate, whereas the left eigenvector gives the partition function $Z$.

In [9]:
ws.render_section('sec_ergodicity_spectrum')

## 2. Spectral Decomposition, Convergence Rate & Ergodicity

How do we know whether iterating $f_{n+1} = f_n P$ will actually converge to the desired equilibrium distribution $\pi(s)$, and how many "warmup" (equilibration) steps are required?

### Spectral Decomposition of the Master Equation
Expanding the initial probability distribution $f_0(s)$ in the basis of left eigenvectors $\phi_\lambda(s)$ of $P$ (where $\phi_\lambda P = \varepsilon_\lambda \phi_\lambda$), the state distribution after $n$ steps is:
$$f_n(s) = \pi(s) + \sum_{\lambda \ge 1} \varepsilon_\lambda^n\, c_\lambda\, \phi_\lambda(s)$$
Because $P(s \to s') \ge 0$ and $\sum_{s'} P(s \to s') = 1$, the **Perron–Frobenius theorem** guarantees that:
- All eigenvalues satisfy $|\varepsilon_\lambda| \le 1$.
- Every stationary equilibrium distribution corresponds to an eigenvalue $\varepsilon = 1$.
- If the Markov chain is **ergodic**, then $\varepsilon_0 = 1$ is **non-degenerate** (unique), and every other eigenvalue satisfies the strict inequality:
  $$1 = \varepsilon_0 > |\varepsilon_1| \ge |\varepsilon_2| \ge \cdots$$

### Geometric Convergence & Warmup Time $\tau_{\text{eq}}$
When $|\varepsilon_\lambda| < 1$ for all $\lambda \ge 1$, the transient terms $\varepsilon_\lambda^n$ decay **geometrically** to zero as $n \to \infty$, so $f_n(s) \to \pi(s)$ from *any* initial state $f_0(s)$. At large $n$, convergence is controlled by the **second-largest eigenvalue in magnitude**, $|\varepsilon_1|$:
$$|f_n(s) - \pi(s)| \sim |\varepsilon_1|^n = \exp\!\left(n \ln|\varepsilon_1|\right) = \exp\!\left(-\frac{n}{\tau_{\text{eq}}}\right)$$
Defining the characteristic **relaxation (or equilibration) time** $\tau_{\text{eq}}$ in units of Markov steps gives:
$$\tau_{\text{eq}} = -\frac{1}{\ln|\varepsilon_1|} \approx \frac{1}{1 - |\varepsilon_1|} \quad \text{(when } |\varepsilon_1| \to 1^-\text{)}$$
If $|\varepsilon_1|$ is very close to $1$ (a small **spectral gap** $1 - |\varepsilon_1| \ll 1$, typical of high free-energy barriers or phase coexistence), $\tau_{\text{eq}}$ becomes enormous and the walk suffers from **slow mixing** (critical or glassy slowing down).

### The Three Conditions for Ergodicity (and Their Failure Modes)
In contrast to deterministic Molecular Dynamics (where proving ergodicity for a general Hamiltonian is notoriously difficult, as shown by the FPU paradox and KAM theorem), **ergodicity of a Markov chain can be proven rigorously** if three conditions hold:
1. **Irreducibility (Connectedness / No Infinite Barriers):** One can move from any state $s$ to any other state $s'$ in a finite number of steps with non-zero probability ($P^k(s \to s') > 0$ for some finite $k$).
   - *Failure Mode 1 (Reducible / Disconnected Space):* If phase space is split into disconnected pockets by infinite energy barriers or restrictive move rules, the eigenvalue $\varepsilon = 1$ becomes **degenerate** (multiple linearly independent stationary distributions), and the final distribution depends permanently on where the walk was initialized!
2. **Aperiodicity (Non-Periodic Transition Rules):** The return times to any state are not restricted to integer multiples of a period $p \ge 2$.
   - *Failure Mode 2 (Bipartite / Periodic Oscillation):* If a particle only hops between nearest neighbors on a bipartite lattice with zero probability of staying in place ($P(s \to s) = 0$), the chain has an eigenvalue $\varepsilon = -1$ (or complex roots of unity $|\varepsilon| = 1$). Then $\varepsilon^n = (-1)^n$ oscillates forever without decaying! Note that in Metropolis MC, nonzero rejection probabilities $P(s \to s) > 0$ automatically break periodicity.
3. **Positive Recurrence (Finite Mean Return Time):** The average number of steps required to return to any state is finite.
   - *Failure Mode 3 (Null Recurrence / "Expanding Universe"):* In an unbounded infinite volume (e.g., an unconfined random walk in $\mathbb{R}^3$), particles can diffuse away to infinity so the mean return time diverges and no normalizable stationary distribution exists. This is never a problem in a finite system or a periodic simulation box.


In [10]:
ws.render_section('concept_markov_convergence')

### Interactive Explorer: Markov Chain Eigenvalues, Relaxation & Ergodicity Traps

Select different 2-state Markov transition matrices $P$ and vary the initial probability $f_0(0)$ to observe geometric convergence inside the spectral envelope $\pi(0) \pm |\varepsilon_1|^n$, slow mixing when $\varepsilon_1 \to 1$, and ergodicity breakdowns under periodicity ($\varepsilon_1 = -1$) or reducibility ($\varepsilon_1 = +1$).

**Key Takeaway:** An ergodic chain has a unique $\varepsilon_0 = 1$ and $|\varepsilon_1| < 1$, guaranteeing geometric decay of initial-state memory on timescale $\tau_{\mathrm{eq}} = -1/\ln|\varepsilon_1|$. If $|\varepsilon_1| = 1$ (either $+1$ from disconnected barriers or $-1$ from bipartite periodicity), the walk never equilibrates to a unique stationary state.

In [11]:
ws.render_section('sec_reliability_example')

## 3. Worked Example: The 2-State Computer Reliability Markov Chain

Following Allen & Tildesley (pp. 110–123) and the lecture notes, consider a computer whose daily reliability follows a 2-state Markov chain between state $1 = \text{Up}$ and state $2 = \text{Down}$:
- If the computer is **Up** today, it has a $60\%$ chance of remaining **Up** tomorrow and a $40\%$ chance of going **Down**.
- If the computer is **Down** today, it has a $70\%$ chance of remaining **Down** tomorrow and a $30\%$ chance of coming back **Up**.

This defines the stochastic transition matrix:
$$P(s \to s') = \begin{pmatrix} 0.6 & 0.4 \\ 0.3 & 0.7 \end{pmatrix}$$

### Step-by-Step Transient Evolution
Suppose on day $1$ the computer is equally likely to be Up or Down:
$$\pi(1) = (0.5,\; 0.5)$$
- **Day 2 distribution:**
  $$\pi(2) = \pi(1) P = (0.5,\; 0.5)\begin{pmatrix} 0.6 & 0.4 \\ 0.3 & 0.7 \end{pmatrix} = \big(0.5(0.6) + 0.5(0.3),\; 0.5(0.4) + 0.5(0.7)\big) = (0.45,\; 0.55)$$
- **Day 3 distribution:**
  $$\pi(3) = \pi(2) P = (0.45,\; 0.55)\begin{pmatrix} 0.6 & 0.4 \\ 0.3 & 0.7 \end{pmatrix} = \big(0.45(0.6) + 0.55(0.3),\; 0.45(0.4) + 0.55(0.7)\big) = (0.435,\; 0.565)$$

### Exact Stationary State $\pi(\infty)$ & Eigenvalues
In the limit $n \to \infty$, $\pi(\infty) = (\pi_{\text{Up}}, \pi_{\text{Dn}})$ satisfies the left eigenvalue equation $\pi = \pi P$ with $\pi_{\text{Up}} + \pi_{\text{Dn}} = 1$:
$$0.6\,\pi_{\text{Up}} + 0.3\,\pi_{\text{Dn}} = \pi_{\text{Up}} \quad \Longrightarrow \quad 0.4\,\pi_{\text{Up}} = 0.3\,\pi_{\text{Dn}} \quad \Longrightarrow \quad \frac{\pi_{\text{Up}}}{\pi_{\text{Dn}}} = \frac{3}{4}$$
Normalizing gives the unique equilibrium distribution:
$$\pi(\infty) = \left(\frac{3}{7},\; \frac{4}{7}\right) \approx (0.4286,\; 0.5714)$$
What about the eigenvalues of $P$? Since $\varepsilon_0 = 1$ and the sum of eigenvalues equals the matrix trace $\operatorname{Tr}(P) = 0.6 + 0.7 = 1.3$:
$$\varepsilon_0 + \varepsilon_1 = 1.3 \quad \Longrightarrow \quad \varepsilon_1 = 0.3$$
Indeed, at every step the deviation from equilibrium $\pi(n) - \pi(\infty)$ shrinks by an exact factor of $\varepsilon_1 = 0.3$:
$$\pi_{\text{Up}}(1) - \frac{3}{7} = \frac{1}{14}, \qquad \pi_{\text{Up}}(2) - \frac{3}{7} = 0.3 \times \frac{1}{14} = \frac{3}{140}, \qquad \pi_{\text{Up}}(3) - \frac{3}{7} = 0.3^2 \times \frac{1}{14} = \frac{9}{1400}$$


In [12]:
ws.render_section('q15_07_perron_frobenius_bounds')

**Q7 · easy** · 💡

For an ergodic Markov chain with transition matrix $P$ and eigenvalues $\varepsilon_0, \varepsilon_1, \varepsilon_2, \dots$, which condition on the eigenvalue spectrum guarantees convergence to a **unique** stationary distribution $\pi(s)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the spectral expansion $f_n(s) = \pi(s) + \sum_{\lambda \ge 1} \varepsilon_\lambda^n c_\lambda \phi_\lambda(s)$ as $n \to \infty$.

</div>
</details>

- [ ] **(A)** A single non-degenerate eigenvalue $\varepsilon_0 = 1$, and all other eigenvalues strictly inside the unit circle ($|\varepsilon_\lambda| < 1$ for all $\lambda \ge 1$).
- [ ] **(B)** All eigenvalues equal to $+1$ ($\varepsilon_\lambda = 1$ for all $\lambda$).
- [ ] **(C)** At least one eigenvalue with $\varepsilon_\lambda > 1$ so that equilibrium states grow exponentially.
- [ ] **(D)** All eigenvalues equal to $0$ ($\varepsilon_\lambda = 0$ for all $\lambda$).

In [13]:
ws.render_section('q15_08_relaxation_time_calculation')

**Q8 · medium** · 💡

A Markov chain sampling a double-well potential has a second-largest eigenvalue $\varepsilon_1 = 0.95$. Calculate the characteristic relaxation (warmup) time $\tau_{\text{eq}} = -1/\ln|\varepsilon_1|$ in Markov steps.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $-1 / \ln(0.95)$.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q15_09_ergodicity_three_conditions')

**Q9 · medium** · 💡

Select **all** conditions that are required to prove that a Markov chain is **ergodic** and converges to a unique stationary distribution:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the three properties listed on slide 4 of the lecture notes.

</div>
</details>

- [ ] **(A)** Irreducibility (no infinite barriers): the system can move from any state to any other state in a finite number of steps with non-zero probability.
- [ ] **(B)** Aperiodicity: the transition rules are non-periodic (e.g., not trapped in deterministic alternating hops on a bipartite lattice).
- [ ] **(C)** Positive recurrence: the average return time to any state is finite (no 'expanding universe' escape to infinity).
- [ ] **(D)** Zero rejection rate: every proposed trial move must be accepted with probability $A = 1$.

In [15]:
ws.render_section('q15_10_reducible_degenerate_eigenvalues')

**Q10 · medium** · 💡

Suppose a simulation of a crystal has an infinite energy barrier separating $k = 3$ disconnected symmetry-broken domains that the MC move set can never cross. How does this ergodicity failure manifest in the eigenvalue spectrum of the transition matrix $P$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each disconnected domain supports its own independent stationary equilibrium distribution.

</div>
</details>

- [ ] **(A)** The unit eigenvalue $\varepsilon = 1$ becomes $3$-fold degenerate, so the limiting distribution $f_\infty$ is not unique and depends on the initial state $f_0$.
- [ ] **(B)** All eigenvalues of $P$ become strictly negative ($\varepsilon_\lambda < 0$).
- [ ] **(C)** The largest eigenvalue exceeds unity ($\varepsilon_0 = 3$).
- [ ] **(D)** The transition matrix $P$ has no eigenvectors at all.

In [16]:
ws.render_section('q15_11_bipartite_periodicity_eigenvalue')

**Q11 · medium** · 💡

A random walker on a bipartite lattice (consisting of sublattices $A$ and $B$) must hop to a nearest neighbor on the opposite sublattice at every step ($P(A \to B) = 1$, $P(B \to A) = 1$, with $P(s \to s) = 0$). If the walker starts entirely on sublattice $A$ ($f_0 = (1, 0)$), what happens as $n \to \infty$ and why?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the eigenvalues of $P = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$ and how $\varepsilon_1^n$ behaves.

</div>
</details>

- [ ] **(A)** $P$ has a second eigenvalue $\varepsilon_1 = -1$ with $|\varepsilon_1| = 1$, so $f_n$ alternates between $(1, 0)$ and $(0, 1)$ forever as $(-1)^n$ without converging to $\pi = (0.5, 0.5)$.
- [ ] **(B)** $f_n$ converges monotonically to $(0.5, 0.5)$ in exactly $2$ steps because $P$ is doubly stochastic.
- [ ] **(C)** $P$ violates row normalization because its diagonal entries are zero.
- [ ] **(D)** The total probability $\sum_s f_n(s)$ decays to zero as $n \to \infty$.

In [17]:
ws.render_section('q15_12_reliability_day2_prob')

**Q12 · easy** · 💡

In the computer reliability Markov chain with transition matrix
$$P = \begin{pmatrix} 0.6 & 0.4 \\ 0.3 & 0.7 \end{pmatrix}$$
suppose the computer is definitely **Up** on day 1: $f_1 = (1.0,\; 0.0)$. What is the probability $f_3(\text{Up})$ that the computer is **Up** on day 3 (after $2$ transitions)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $f_2 = f_1 P = (0.6, 0.4)$ and then $f_3 = f_2 P$.

</div>
</details>

_Numeric answer required._

In [18]:
ws.render_section('q15_13_reliability_stationary_up')

**Q13 · easy** · 💡

For the computer reliability Markov chain $P = \begin{pmatrix} 0.6 & 0.4 \\ 0.3 & 0.7 \end{pmatrix}$, what is the exact stationary probability $\pi_{\text{Up}}(\infty)$ that the computer is **Up** after many days? (Enter a decimal to at least 3 significant figures.)


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve $\pi = \pi P$ with $\pi_{\text{Up}} + \pi_{\text{Dn}} = 1$, which gives $\pi_{\text{Up}} = 3/7$.

</div>
</details>

_Numeric answer required._

In [19]:
ws.render_section('q15_14_reliability_second_eigenvalue')

**Q14 · medium** · 💡

What is the second eigenvalue $\varepsilon_1$ of the computer reliability transition matrix $P = \begin{pmatrix} 0.6 & 0.4 \\ 0.3 & 0.7 \end{pmatrix}$ that governs the geometric decay rate $|f_n - \pi(\infty)| \propto \varepsilon_1^n$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use $\varepsilon_0 + \varepsilon_1 = \operatorname{Tr}(P) = 0.6 + 0.7$ with $\varepsilon_0 = 1$.

</div>
</details>

_Numeric answer required._

In [20]:
ws.render_section('q15_15_general_2state_stationary')

**Q15 · medium** · 💡

Suppose you upgrade the computer's hardware so its daily transition matrix becomes:
$$P = \begin{pmatrix} 0.85 & 0.15 \\ 0.45 & 0.55 \end{pmatrix}$$
What is the new steady-state probability $\pi_{\text{Up}}(\infty)$ of being **Up**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Balance the flux between Up and Down: $0.15\, \pi_{\text{Up}} = 0.45\, \pi_{\text{Dn}}$ with $\pi_{\text{Up}} + \pi_{\text{Dn}} = 1$.

</div>
</details>

_Numeric answer required._

In [21]:
ws.render_section('q15_16_general_2state_eigenvalue')

**Q16 · medium** · 💡

For the upgraded reliability matrix $P = \begin{pmatrix} 0.85 & 0.15 \\ 0.45 & 0.55 \end{pmatrix}$ in the previous question, what is its second eigenvalue $\varepsilon_1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\operatorname{Tr}(P) - 1 = 0.85 + 0.55 - 1$.

</div>
</details>

_Numeric answer required._

In [22]:
ws.render_section('sec_detailed_balance')

## 4. Detailed Balance (Microscopic Reversibility) & Rejection

In our reliability example, we were *given* the transition matrix $P(s \to s')$ and had to solve for its unknown stationary distribution $\pi(s)$. In **Metropolis Monte Carlo**, the problem is inverted:
- We **know** the target equilibrium distribution we want to sample—for example, the canonical Boltzmann distribution:
  $$\pi(s) = \frac{1}{Z} e^{-\beta E(s)}$$
- We want to **construct** a transition matrix $P(s \to s')$ that is guaranteed to have $\pi(s)$ as its unique stationary eigenvector ($\pi P = \pi$).

### The Three Key Concepts of the Metropolis Algorithm
1. **Sample phase space by using an ergodic random walk.**
2. **Guarantee the desired equilibrium state $\pi(s)$ by enforcing detailed balance.**
3. **Achieve detailed balance by rejecting trial moves.**

### Detailed Balance vs. Global Balance
The global stationarity condition (master equation in steady state) requires that the total probability flowing *into* state $s'$ equals $\pi(s')$:
$$\pi(s') = \sum_{s} \pi(s)\, P(s \to s')$$
Solving this coupled sum over all $s$ simultaneously is difficult. Instead, we impose a stronger pairwise condition called **detailed balance** (or **microscopic reversibility**), which equates the steady-state transition rate $K(s \to s') \equiv \pi(s)P(s \to s')$ from $s$ to $s'$ with the reverse rate $K(s' \to s)$ from $s'$ to $s$:
$$\pi(s)\, P(s \to s') = \pi(s')\, P(s' \to s) \qquad \forall\, s, s'$$

### Proof that Detailed Balance Implies Stationarity ($\pi = \pi P$)
Summing both sides of the detailed balance equation over all states $s$ and applying row normalization $\sum_s P(s' \to s) = 1$ immediately proves that $\pi(s)$ is a stationary eigenvector of $P$:
$$\sum_{s} \pi(s)\, P(s \to s') = \sum_{s} \pi(s')\, P(s' \to s) = \pi(s') \underbrace{\sum_{s} P(s' \to s)}_{=\, 1} = \pi(s')$$
- **Sufficient, not strictly necessary:** Detailed balance is a *sufficient* condition for $\pi = \pi P$ (and guarantees that $P$ has real eigenvalues because $\pi^{1/2} P \pi^{-1/2}$ is symmetric), though as we will see in Lesson 16, sequential menus of moves can satisfy global balance without satisfying detailed balance.
- **Self-transitions from rejections:** Because off-diagonal moves to higher-energy states are sometimes rejected, there is a nonzero probability of remaining in the same state $s$:
  $$P(s \to s) = 1 - \sum_{s' \ne s} P(s \to s')$$


In [23]:
ws.render_section('q15_17_detailed_balance_definition')

**Q17 · easy** · 💡

Which equation expresses the **detailed balance** condition for a Markov chain with transition matrix $P(s \to s')$ and target equilibrium distribution $\pi(s)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Equate the joint one-step transition flux $K(s \to s')$ from $s$ to $s'$ with the reverse flux $K(s' \to s)$.

</div>
</details>

- [ ] **(A)** $\pi(s)\, P(s \to s') = \pi(s')\, P(s' \to s)$ for every pair of states $(s, s')$
- [ ] **(B)** $P(s \to s') = P(s' \to s)$ for every pair of states $(s, s')$
- [ ] **(C)** $\pi(s) = \pi(s')$ for every pair of states $(s, s')$
- [ ] **(D)** $\sum_{s'} \pi(s')\, P(s \to s') = 1$ for every state $s$

In [24]:
ws.render_section('q15_18_detailed_balance_canonical_ratio')

**Q18 · easy** · 💡

When sampling the canonical Boltzmann distribution $\pi(s) = \frac{1}{Z}e^{-\beta E(s)}$, what ratio of forward-to-reverse transition probabilities $P(s \to s') / P(s' \to s)$ is required by detailed balance?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Rearrange $\pi(s)P(s \to s') = \pi(s')P(s' \to s)$ for $P(s \to s') / P(s' \to s)$.

</div>
</details>

- [ ] **(A)** $\dfrac{P(s \to s')}{P(s' \to s)} = \dfrac{\pi(s')}{\pi(s)} = \exp\big(-\beta[E(s') - E(s)]\big)$
- [ ] **(B)** $\dfrac{P(s \to s')}{P(s' \to s)} = \dfrac{\pi(s)}{\pi(s')} = \exp\big(+\beta[E(s') - E(s)]\big)$
- [ ] **(C)** $\dfrac{P(s \to s')}{P(s' \to s)} = 1$
- [ ] **(D)** $\dfrac{P(s \to s')}{P(s' \to s)} = \dfrac{E(s)}{E(s')}$

In [25]:
ws.render_section('q15_19_detailed_vs_global_balance')

**Q19 · medium** · 💡

What is the exact logical relationship between **detailed balance** ($\pi(s)P(s \to s') = \pi(s')P(s' \to s)$) and **global stationary balance** ($\pi(s') = \sum_s \pi(s)P(s \to s')$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider whether summing detailed balance over $s$ gives global balance, and whether probability currents can circulate in closed loops at steady state.

</div>
</details>

- [ ] **(A)** Detailed balance is a **sufficient** condition for global stationary balance, but it is **not strictly necessary** (a chain with steady-state probability currents around cycles can still preserve $\pi = \pi P$).
- [ ] **(B)** Detailed balance is both necessary and sufficient; no Markov chain can have $\pi = \pi P$ without satisfying detailed balance.
- [ ] **(C)** Global stationary balance implies detailed balance, but detailed balance does not imply global balance.
- [ ] **(D)** Detailed balance only applies to 2-state chains and fails for $N > 2$ states.

In [26]:
ws.render_section('q15_20_self_transition_probability')

**Q20 · easy** · 💡

A system in state $s$ can propose moves to three neighboring states $s_1, s_2, s_3$. Suppose the actual off-diagonal transition probabilities (proposal $\times$ acceptance) are $P(s \to s_1) = 0.25$, $P(s \to s_2) = 0.15$, and $P(s \to s_3) = 0.05$. What is the self-transition probability $P(s \to s)$ that the system remains in state $s$ due to move rejection?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use $P(s \to s) = 1 - \sum_{s' \ne s} P(s \to s')$.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q15_21_three_key_concepts_metropolis')

**Q21 · easy** · 💡

Select the **three key concepts** that form the foundation of the Metropolis algorithm (as highlighted on slide 7 of the lecture notes):

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall how Metropolis explores phase space, sets the stationary distribution, and enforces the transition ratio.

</div>
</details>

- [ ] **(A)** Sample phase space by using an ergodic random walk.
- [ ] **(B)** Determine the equilibrium state $\pi(s)$ by imposing detailed balance.
- [ ] **(C)** Achieve detailed balance by rejecting trial moves.
- [ ] **(D)** Integrate Hamilton's equations of motion using a symplectic Verlet time step.

In [28]:
ws.render_section('sec_metropolis_hastings')

### Factoring Transition Probability into Proposal and Acceptance
To construct $P(s \to s')$ in practice, the **Metropolis–Hastings method** factors each off-diagonal transition ($s' \ne s$) into two stages:
$$P(s \to s') = T(s \to s')\, A(s \to s')$$
1. **Sampling / Proposal distribution $T(s \to s')$:** The conditional probability of *generating* a trial state $s'$ given current state $s$ (with $\sum_{s'} T(s \to s') = 1$).
2. **Acceptance probability $A(s \to s') \in [0, 1]$:** The conditional probability of *accepting* the proposed move $s \to s'$. If rejected (with probability $1 - A(s \to s')$), the system stays at $s_{n+1} = s$.

Substituting $P(s \to s') = T(s \to s') A(s \to s')$ into the detailed balance condition $\pi(s) P(s \to s') = \pi(s') P(s' \to s)$ gives:
$$\frac{A(s \to s')}{A(s' \to s)} = \frac{T(s' \to s)\, \pi(s')}{T(s \to s')\, \pi(s)} \equiv q(s \to s')$$

### The Generalized Metropolis–Hastings Acceptance Formula
To maximize the acceptance of moves while respecting $0 \le A \le 1$, Metropolis et al. (1953) and Hastings (1970) set the larger of $A(s \to s')$ and $A(s' \to s)$ to $1$, yielding:
$$A_M(s \to s') = \min\!\left\{1,\; \frac{T(s' \to s)\, \pi(s')}{T(s \to s')\, \pi(s)}\right\} = \min\{1, q\}$$
- **Crucial Property — Partition Function $Z$ Cancels!** Because $\pi(s)$ enters only through the ratio $\pi(s')/\pi(s) = e^{-\beta[E(s') - E(s)]}$, the intractable canonical partition function $Z = \sum_s e^{-\beta E(s)}$ **cancels out completely**!
- **Classic Metropolis (Symmetric Proposal $T(s \to s') = T(s' \to s)$):** When the proposal distribution is symmetric (e.g., uniform displacement in a cube around the current position), $T(s' \to s)/T(s \to s') = 1$ and the acceptance probability simplifies to:
  $$A_M(s \to s') = \min\!\left\{1,\; \frac{\pi(s')}{\pi(s)}\right\} = \min\!\left\{1,\; e^{-\beta \Delta E}\right\}, \qquad \Delta E = E(s') - E(s)$$

### Practical Implementation of the Rejection Step
Given $\Delta E = E(s') - E(s)$ (for a symmetric proposal):
- If $\Delta E \le 0$ (the trial move lowers or preserves energy), $e^{-\beta \Delta E} \ge 1 \implies A_M = 1$: **always accept**.
- If $\Delta E > 0$ (the trial move increases energy), draw a uniform random number $u_n \in (0, 1)$:
  - If $u_n < e^{-\beta \Delta E}$, **accept** the move ($s_{n+1} = s'$).
  - If $u_n > e^{-\beta \Delta E}$, **reject** the move ($s_{n+1} = s$).

### Alternative Solutions to Detailed Balance: Barker Rule & Peskun's Theorem
Detailed balance only fixes the *ratio* $A(s \to s')/A(s' \to s) = q$. Other valid choices exist! Most famously, **Barker's (1965) / two-state Heat-Bath rule** sets:
$$A_B(s \to s') = \frac{q}{1 + q} = \frac{T(s' \to s)\pi(s')}{T(s \to s')\pi(s) + T(s' \to s)\pi(s')} \xrightarrow{T\text{ sym}} \frac{\pi(s')}{\pi(s) + \pi(s')} = \frac{1}{1 + e^{\beta \Delta E}}$$
Notice that at $\Delta E = 0$ ($q = 1$), Barker accepts only $A_B(1) = \frac{1}{2}$ of neutral moves, whereas Metropolis accepts $A_M(1) = 1$. In fact, for every $q > 0$:
$$A_M(q) = \min(1, q) > \frac{q}{1 + q} = A_B(q)$$
By **Peskun's Theorem (1973)**, because the Metropolis rule $A_M$ dominates every other valid pairwise acceptance rule element-by-element on all off-diagonal transitions ($P_M(s \to s') \ge P(s \to s')$ for all $s \ne s'$), **Metropolis minimizes the asymptotic variance of MCMC estimators** for any fixed proposal matrix $T$.


In [29]:
ws.render_section('concept_metropolis_vs_barker')

### Interactive Explorer: Metropolis vs. Barker Acceptance & Peskun's Theorem

Adjust the dimensionless energy change $\beta \Delta E$ and uniform random number $u_n \in (0, 1)$ to visualize the Metropolis acceptance curve $A_M = \min(1, e^{-\beta\Delta E})$, the Barker curve $A_B = 1/(1 + e^{\beta\Delta E})$, and the Peskun efficiency gap $A_M - A_B$.

**Key Takeaway:** For any energy change $\Delta E$, the Metropolis rule $A_M = \min(1, e^{-\beta\Delta E})$ gives a strictly higher acceptance probability than Barker's rule $A_B = (1 + e^{\beta\Delta E})^{-1}$, maximizing off-diagonal state transitions and minimizing statistical variance.

In [30]:
ws.render_section('q15_22_why_partition_function_not_needed')

**Q22 · easy** · 💡

Why can the Metropolis algorithm sample the canonical distribution $\pi(s) = \frac{1}{Z}e^{-\beta E(s)}$ without ever computing the partition function $Z = \sum_s e^{-\beta E(s)}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect how $\pi(s)$ appears inside the acceptance formula $A(s \to s') = \min\left(1, \frac{T(s'\to s)\pi(s')}{T(s\to s')\pi(s)}\right)$.

</div>
</details>

- [ ] **(A)** Because $A(s \to s')$ depends only on the ratio $\pi(s')/\pi(s)$, so the state-independent normalization constant $1/Z$ cancels out between numerator and denominator.
- [ ] **(B)** Because $Z = 1$ identically at all temperatures in the canonical ensemble.
- [ ] **(C)** Because the proposal matrix $T(s \to s')$ is normalized by $Z$.
- [ ] **(D)** Because rejected moves automatically compute $Z$ at the end of the run.

In [31]:
ws.render_section('q15_23_classic_metropolis_acceptance_calc')

**Q23 · easy** · 💡

In a classic Metropolis simulation with a symmetric proposal ($T(s \to s') = T(s' \to s)$) at temperature $k_B T = 0.050\text{ eV}$, a trial move increases the system energy by $\Delta E = E(s') - E(s) = +0.075\text{ eV}$. What is the Metropolis acceptance probability $A_M(s \to s')$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\beta \Delta E = \Delta E / (k_B T) = 0.075 / 0.050 = 1.5$ and evaluate $\exp(-1.5)$.

</div>
</details>

_Numeric answer required._

In [32]:
ws.render_section('q15_24_downhill_move_acceptance')

**Q24 · easy** · 💡

If a symmetric-proposal Metropolis move **lowers** the potential energy by $\Delta E = E(s') - E(s) = -0.12\text{ eV}$ at $k_B T = 0.025\text{ eV}$, what is the acceptance probability $A_M(s \to s')$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall that $A_M = \min(1, e^{-\beta \Delta E})$ and probabilities cannot exceed $1$.

</div>
</details>

_Numeric answer required._

In [33]:
ws.render_section('q15_25_udrn_rejection_criterion')

**Q25 · easy** · 💡

A trial move has $\beta \Delta E = +1.0$, giving $e^{-\beta \Delta E} \approx 0.3679$. The pseudorandom number generator returns a uniform variate $u_n = 0.52$ on $(0, 1)$. What does the Metropolis algorithm do?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $u_n$ against $e^{-\beta \Delta E}$ using slide 11 of the lecture notes.

</div>
</details>

- [ ] **(A)** Reject the move (since $u_n > e^{-\beta \Delta E}$), keeping $s_{n+1} = s_n$.
- [ ] **(B)** Accept the move (since $u_n > e^{-\beta \Delta E}$), setting $s_{n+1} = s'$.
- [ ] **(C)** Redraw $u_n$ repeatedly until $u_n < e^{-\beta \Delta E}$.
- [ ] **(D)** Accept half of the displacement vector $\Delta \mathbf{r} / 2$.

In [34]:
ws.render_section('q15_26_asymmetric_metropolis_hastings_calc')

**Q26 · medium** · 💡

In a generalized Metropolis–Hastings move with an **asymmetric** proposal distribution $T(s \to s') \ne T(s' \to s)$, suppose:
- The forward proposal probability is $T(s \to s') = 0.40$,
- The reverse proposal probability is $T(s' \to s) = 0.20$, and
- The Boltzmann weight ratio is $\pi(s')/\pi(s) = e^{-\beta \Delta E} = 1.50$.

What is the Metropolis–Hastings acceptance probability $A_M(s \to s')$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $A_M(s \to s') = \min\!\left(1,\, \frac{T(s' \to s)\pi(s')}{T(s \to s')\pi(s)}\right)$.

</div>
</details>

_Numeric answer required._

In [35]:
ws.render_section('q15_27_barker_acceptance_calc')

**Q27 · medium** · 💡

For a symmetric proposal $T(s \to s') = T(s' \to s)$ and an uphill energy change $\beta \Delta E = \ln(3) \approx 1.0986$ (so that $\pi(s')/\pi(s) = e^{-\beta \Delta E} = 1/3$), what is the acceptance probability $A_B(s \to s')$ under **Barker's acceptance rule** $A_B = \frac{\pi(s')}{\pi(s) + \pi(s')}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $q / (1 + q)$ for $q = 1/3$.

</div>
</details>

_Numeric answer required._

In [36]:
ws.render_section('q15_28_peskun_theorem_advantage')

**Q28 · medium** · 💡

Consider a move between two degenerate states of identical energy ($\Delta E = 0$, symmetric proposal $T$). By how much does the Metropolis acceptance probability $A_M$ exceed the Barker acceptance probability $A_B$, and what does **Peskun's theorem** conclude from $A_M(q) \ge A_B(q)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $A_M(1) = \min(1, 1)$ with $A_B(1) = 1/(1+1)$.

</div>
</details>

- [ ] **(A)** $A_M = 1.0$ vs. $A_B = 0.5$ (twice as high at $\Delta E = 0$); by maximizing all off-diagonal transition probabilities $P(s \to s')$, Metropolis minimizes the asymptotic variance of sample averages for a fixed proposal $T$.
- [ ] **(B)** $A_M = 0.5$ vs. $A_B = 1.0$; Barker's rule is always twice as fast as Metropolis.
- [ ] **(C)** $A_M$ and $A_B$ are identical at $\Delta E = 0$ and only differ when $\Delta E \to \infty$.
- [ ] **(D)** Metropolis violates detailed balance at $\Delta E = 0$, whereas Barker satisfies it.

In [37]:
ws.render_section('q15_29_infinite_temperature_limit')

**Q29 · easy** · 💡

For a finite-energy system with a symmetric proposal $T(s \to s') = T(s' \to s)$ inside a periodic box, how does the Metropolis acceptance probability $A_M(s \to s') = \min(1, e^{-\beta \Delta E})$ behave in the high-temperature limit $T \to \infty$ ($\beta \to 0^+$) and low-temperature limit $T \to 0^+$ ($\beta \to \infty$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $e^{-\beta \Delta E}$ as $\beta \to 0$ and as $\beta \to \infty$.

</div>
</details>

- [ ] **(A)** As $T \to \infty$ ($\beta \to 0$), $A_M \to 1$ for all finite-energy moves (unbiased random walk); as $T \to 0$ ($\beta \to \infty$), $A_M \to 0$ for uphill moves ($\Delta E > 0$) and $A_M = 1$ for downhill moves ($\Delta E \le 0$, greedy quench).
- [ ] **(B)** As $T \to \infty$, $A_M \to 0$; as $T \to 0$, $A_M \to 1$ for all moves.
- [ ] **(C)** As $T \to \infty$, $A_M \to 0.5$; as $T \to 0$, $A_M \to 0.5$.
- [ ] **(D)** Temperature has no effect on $A_M$ because $\beta$ cancels out with $Z$.

In [38]:
ws.render_section('q15_30_hard_sphere_metropolis')

**Q30 · easy** · 💡

In the original 1953 Metropolis et al. study of a **hard-sphere liquid** ($v(r) = \infty$ if $r < \sigma$, and $v(r) = 0$ if $r \ge \sigma$), what does the canonical acceptance rule $A_M(s \to s') = \min(1, e^{-\beta \Delta E})$ reduce to?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In a hard-sphere fluid Starting from a valid non-overlapping state, a trial move either creates an overlap ($\Delta E = +\infty$) or no overlap ($\Delta E = 0$).

</div>
</details>

- [ ] **(A)** Accept the move with probability $A_M = 1$ if no spheres overlap ($r_{ij}' \ge \sigma$), and reject with probability $A_M = 0$ if any pair overlaps ($r_{ij}' < \sigma$), independent of temperature.
- [ ] **(B)** Accept every move with probability $0.5$ regardless of overlaps.
- [ ] **(C)** Accept overlapping moves with probability $e^{-\beta \sigma}$.
- [ ] **(D)** Hard spheres cannot be simulated with Metropolis MC because the force $-\nabla v(r)$ is a delta function.

In [39]:
ws.render_section('sec_single_particle_and_bugs')

### How to Sample Continuous Coordinates: The Cube Move
In a continuous fluid or solid of $N$ atoms, how should we propose a trial move $s \to s'$?
As shown on slide 12 of the lecture notes, we select **one particle $i$ at a time** and displace its position uniformly inside a cube of side length $\Delta$ centered on its current coordinate $\mathbf{r}_i$:
$$\mathbf{r}_i^{\text{new}} = \mathbf{r}_i^{\text{old}} + \Delta \cdot (\text{sprng}() - 0.5)$$
where $\text{sprng}()$ returns $3$ independent uniform random numbers on $(0, 1)$, so each Cartesian component shifts by $\delta x, \delta y, \delta z \in [-\Delta/2, +\Delta/2]$. Inside the cube, $T(\mathbf{r}_i \to \mathbf{r}_i') = 1/\Delta^3 = \text{constant}$ (symmetric!).

### Why Move One Particle at a Time Instead of All $N$ Particles?
1. **$\mathcal{O}(1)$ Local Energy Difference:** When only particle $i$ moves ($\mathbf{r}_i \to \mathbf{r}_i'$ while $\mathbf{r}_j' = \mathbf{r}_j$ for all $j \ne i$), all pair interactions $v(r_{jk})$ not involving particle $i$ cancel out of $\Delta V = V(s') - V(s)$:
   $$A(s \to s') = \min\!\left\{1,\; \exp\!\left[-\beta \sum_{j \ne i} \big(v(|\mathbf{r}_i' - \mathbf{r}_j|) - v(|\mathbf{r}_i - \mathbf{r}_j|)\big)\right]\right\}$$
   For a short-range potential with cutoff $r_c$, evaluating this sum requires checking only the $\mathcal{O}(1)$ neighbors inside the cutoff sphere of atom $i$, rather than $\mathcal{O}(N^2)$ pairs!
2. **Much Higher Acceptance Ratio for the Same Displacement $\Delta$:** If all $N$ particles are displaced simultaneously by cube width $\Delta$, the typical energy change scales as $\Delta E \sim \mathcal{O}(\sqrt{N})$ (or $\mathcal{O}(N)$), forcing the acceptance rate $e^{-\beta \Delta E}$ to plummet exponentially with $N$ unless $\Delta$ is shrunk as $\Delta \propto N^{-1/2}$.

### Structure of a General Monte Carlo Code (Slide 13)
```python
initstate(s_old)
E_old = action(s_old)                  # Initialize the state and energy
while True:
    sample(s_old, s_new, T_new, 1)     # Propose s_new and get T_new = T(s_old -> s_new)
    E_new = action(s_new)              # Compute trial energy/action
    sample(s_new, s_old, T_old, 0)     # Find reverse prob T_old = T(s_new -> s_old) without moving!
    A = exp(-E_new + E_old) * (T_old / T_new)
    if A > sprng():                    # Accept the move with prob min(1, A)
        s_old = s_new
        E_old = E_new
        naccept = naccept + 1
    averages(s_old)                    # Collect statistics AFTER the if-block (counts rejected states too!)
```

### The Two Classic Errors in MCMC Implementations (Slide 14)
1. **Violating Proposal Reversibility ($T(s \to s') > 0$ while $T(s' \to s) = 0$):**
   If you can propose a move from $s$ to $s'$, **the reverse move from $s'$ to $s$ must also be possible!** For example, if a cube move is asymmetrically defined as $x' \in [x, x + \Delta]$ (only positive shifts) or if $\Delta(s)$ changes with state without overlapping $s$ from $s'$, then $T(s' \to s) = 0$ and detailed balance $\pi(s)T(s \to s')A(s \to s') = \pi(s')T(s' \to s)A(s' \to s)$ is broken unless $A(s \to s') = 0$. You should explicitly test that `sample(s_new, s_old, T_old, 0)` returns $T_{\text{old}} > 0$.
2. **Forgetting that Accepted and Rejected States Count the Same!**
   Notice in the code above that `averages(s_old)` is called **outside** the `if A > sprng():` block! When a trial move $s_n \to s'$ is **rejected**, the Markov chain advances to step $n+1$ with $s_{n+1} = s_n$, and **`s_old` must be accumulated into all ensemble averages again with full weight**. If you accidentally put `averages(s_old)` *inside* the `if` statement (accumulating only when a move is accepted), low-energy states—which reject uphill trial moves most often—are systematically under-counted, biasing $\langle E \rangle$ to artificially high values!


In [40]:
ws.render_section('concept_rejected_state_bug')

### Interactive Demo: Why Omitting Rejected States Destroys the Boltzmann Distribution

Vary the trial displacement width $\Delta$ and inverse temperature $\beta$ for a particle in a 1D harmonic trap $V(x) = \frac{1}{2}x^2$. Compare the exact Boltzmann distribution $\pi(x) \propto e^{-\beta V(x)}$ against a correct MCMC accumulator (which re-counts $s_{n+1} = s_n$ on rejection) and the buggy accumulator (which only records accepted moves).

**Key Takeaway:** At the bottom of a potential well ($x \approx 0$), almost every large trial displacement goes uphill in energy and is rejected. Counting only accepted moves starves the ground state of its rightful residence time $P(s \to s) = 1 - \sum_{s'\ne s} P(s \to s')$, flattening the distribution and artificially inflating $\langle E \rangle$.

In [41]:
ws.render_section('q15_31_single_particle_cube_volume')

**Q31 · easy** · 💡

In a 3D canonical Monte Carlo simulation, a single particle at $\mathbf{r}_i$ is displaced uniformly inside a cube of side length $\Delta = 0.50\text{ Å}$ via $\mathbf{r}_i' = \mathbf{r}_i + \Delta(\mathbf{u} - 0.5)$. What is the constant proposal probability density $T(\mathbf{r}_i \to \mathbf{r}_i') = 1/\Delta^3$ (in $\text{Å}^{-3}$) for any point $\mathbf{r}_i'$ inside that cube?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $1 / (0.50)^3$.

</div>
</details>

_Numeric answer required._

In [42]:
ws.render_section('q15_32_why_move_one_particle')

**Q32 · medium** · 💡

Why do standard canonical Monte Carlo simulations of dense fluids move **one particle at a time** rather than displacing all $N$ particles simultaneously in a single trial step?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider both the computational cost of $\Delta V = V(s') - V(s)$ with a cutoff and how the acceptance ratio behaves when $N$ particles move at once.

</div>
</details>

- [ ] **(A)** Moving one particle requires computing only $\mathcal{O}(1)$ local pair-energy differences within the cutoff range and maintains a much larger acceptance ratio for a given step size $\Delta$.
- [ ] **(B)** Moving more than one particle at a time violates the Markov property and detailed balance.
- [ ] **(C)** Moving all $N$ particles simultaneously always gives an acceptance probability of $A = 1.0$, which prevents rejection.
- [ ] **(D)** Single-particle moves allow exact calculation of real-time phonon dispersion curves.

In [43]:
ws.render_section('q15_33_mc_code_t_old_over_t_new')

**Q33 · medium** · 💡

In the general Monte Carlo pseudocode from slide 13:
```python
sample(s_old, s_new, T_new, 1)
E_new = action(s_new)
sample(s_new, s_old, T_old, 0)
A = exp(-E_new + E_old) * (T_old / T_new)
```
What is the purpose of the second call `sample(s_new, s_old, T_old, 0)` with flag `0`?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at how `T_old` is used in `A = exp(-E_new + E_old) * (T_old / T_new)`.

</div>
</details>

- [ ] **(A)** It evaluates the reverse proposal probability $T_{\text{old}} = T(s_{\text{new}} \to s_{\text{old}})$ without generating a new state, which is needed in the numerator of $A(s_{\text{old}} \to s_{\text{new}})$ whenever $T$ is asymmetric.
- [ ] **(B)** It immediately undoes the move and resets the coordinates back to `s_old` before testing acceptance.
- [ ] **(C)** It sets the temperature $T_{\text{old}}$ to absolute zero ($0\text{ K}$).
- [ ] **(D)** It computes the partition function $Z$ of the old configuration.

In [44]:
ws.render_section('q15_34_bug_omitting_rejected_states_numeric')

**Q34 · medium** · 💡

Suppose a post-equilibration MCMC trajectory runs for $5$ steps starting at energy $E_1 = 1.0\text{ eV}$:
- Step 1: starts at $E_1 = 1.0\text{ eV}$.
- Step 2: proposes $E' = 4.0\text{ eV}$ $\to$ **REJECTED** ($s_2 = s_1$).
- Step 3: proposes $E' = 4.0\text{ eV}$ $\to$ **REJECTED** ($s_3 = s_2$).
- Step 4: proposes $E' = 2.0\text{ eV}$ $\to$ **ACCEPTED** ($s_4$ has $E_4 = 2.0\text{ eV}$).
- Step 5: proposes $E' = 5.0\text{ eV}$ $\to$ **REJECTED** ($s_5 = s_4$).

What is the **correct** 5-step MCMC sample average of the energy $\langle E \rangle = \frac{1}{5}\sum_{n=1}^5 E(s_n)$ (in $\text{eV}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Write out the sequence of actual Markov states $(E(s_1), E(s_2), E(s_3), E(s_4), E(s_5)) = (1.0, 1.0, 1.0, 2.0, 2.0)$ and average all 5 values.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q15_35_bug_omitting_rejected_bias_direction')

**Q35 · medium** · 💡

If a student places `averages(s_old)` *inside* the `if A > sprng():` block so that properties are accumulated **only when a trial move is accepted**, what systematic error occurs in the measured canonical energy distribution $P(E)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Ask which states experience the highest rejection rate: deep energy minima or high-energy states?

</div>
</details>

- [ ] **(A)** Low-energy states (which reject a large fraction of uphill trial moves) are systematically under-weighted relative to high-energy states, shifting $P(E)$ and $\langle E \rangle$ toward artificially high energies.
- [ ] **(B)** High-energy states are under-weighted, causing the system to freeze into the $T = 0$ ground state.
- [ ] **(C)** No bias occurs in $\langle E \rangle$; it only reduces the number of samples and slightly increases the statistical error bar.
- [ ] **(D)** The acceptance ratio becomes greater than $1.0$.

In [46]:
ws.render_section('q15_36_bug_non_reversible_proposal')

**Q36 · medium** · 💡

A student writes a trial move in 1D as `x_new = x_old + Delta * sprng()` (forgetting the `- 0.5` offset, where `sprng()` $\in (0, 1)$), and then uses the standard symmetric acceptance test `A = min(1, exp(-beta * (E_new - E_old)))`. Which fundamental MCMC rule is violated?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If $x_{\text{new}} > x_{\text{old}}$, can this proposal ever generate $x_{\text{old}}$ starting from $x_{\text{new}}$?

</div>
</details>

- [ ] **(A)** Proposal reversibility: $T(x \to x') > 0$ for $x' > x$, but the reverse proposal probability $T(x' \to x) = 0$, violating detailed balance and driving an unphysical one-way drift.
- [ ] **(B)** Row normalization: $\int T(x \to x')\,dx'$ is no longer equal to $1$.
- [ ] **(C)** Energy conservation: potential energy is no longer a function of $x$.
- [ ] **(D)** Nothing is violated as long as periodic boundary conditions are used.

In [47]:
ws.render_section('sec_step_size_and_md_vs_mc')

### How to Choose the Maximum Displacement Step Size $\Delta$
In a single-particle cube move $\mathbf{r}_i' = \mathbf{r}_i + \Delta(\mathbf{u} - 0.5)$, the step size $\Delta$ controls the average **acceptance ratio** ($\text{a.r.}$):
- **If $\Delta \to 0$ (tiny steps):** $\Delta E \to 0$, so the acceptance ratio approaches $\text{a.r.} \to 1.0$ ($100\%$). However, the particles barely move at each step, so phase-space diffusion $\langle \Delta r^2 \rangle \propto \Delta^2$ is extremely slow and serial correlation time $\kappa$ is huge!
- **If $\Delta \to \infty$ (huge steps):** Almost every trial move lands on top of another atom's repulsive core ($\Delta E \gg k_B T$), so the acceptance ratio plummets to $\text{a.r.} \to 0$. The system stays frozen in the same state for thousands of steps!
- **Practical Rule (Slide 15):**
  - **Always measure the acceptance ratio!** Keep it inside the safe operating window:
    $$0.1 < \text{a.r.} < 0.9$$
  - **Rule of thumb:** Adjust $\Delta$ during equilibration so that the acceptance ratio is roughly $\text{a.r.} \approx 0.5$.
  - **Liquid Argon Benchmark (Slides 15–16):** In dense Lennard-Jones liquid Argon (Figs. 1–4 of the lecture notes), the $50\%$ acceptance rule corresponds to $\Delta/\sigma \approx 0.16$. However, measuring the 100-step mean-squared displacement $\langle (\mathbf{r}(i) - \mathbf{r}(i+100))^2 \rangle$ (Fig. 3) reveals that **a $20\%$ acceptance ratio ($\Delta/\sigma \approx 0.36$) actually achieves significantly faster spatial diffusion than a $50\%$ acceptance ratio!**
  - **Robustness of Local Quantities (Slide 16):** Meanwhile, the variance of a local quantity like the total potential energy $\sigma_V$ (Fig. 4) is relatively insensitive to step size across a broad plateau $0.12 \le \Delta/\sigma \le 0.42$. *MC is a robust method—you don't need to fine-tune $\Delta$ to three decimal places!*

### Comprehensive Comparison: Molecular Dynamics (MD) vs. Monte Carlo (MC)
Which method is better—MD or MC? As summarized on slide 18, **you need both!**

| Feature / Capability | Molecular Dynamics (MD) | Metropolis Monte Carlo (MC) |
| :--- | :--- | :--- |
| **Real-time physical dynamics** | **Yes** — integrates Newton's equations to compute transport coefficients, time correlations $C(t)$, and vibrations. | **No** — generates stochastic Markov "kinetics" (useful for long-term diffusive processes in KMC, but not inertial dynamics). |
| **Time-step / discretization error** | **Yes** — finite $\Delta t$ causes symplectic shadow-Hamiltonian drift $\mathcal{O}(\Delta t^2)$ or $\mathcal{O}(\Delta t^4)$. | **Exact!** — no time-step truncation error for any step size $\Delta > 0$! |
| **Force evaluation required?** | **Yes** — requires $-\nabla V(\mathbf{R})$ at every step. | **No** — Classic Metropolis needs only energy differences $\Delta V$ (simpler code, handles discontinuous hard spheres/square wells). |
| **Natural ensemble** | Microcanonical ($NVE$); requires thermostats/barostats for $NVT$/$NPT$. | Direct, exact simulation of **canonical ($NVT$)**, isothermal-isobaric ($NPT$), and **grand canonical ($\mu VT$)** ensembles. |
| **Discrete degrees of freedom** | **No** — requires continuous, differentiable coordinates. | **Yes** — naturally handles Ising spins, lattice gas models, alloy site swaps, and quantum Monte Carlo. |
| **Ergodicity & non-physical moves** | Can be trapped by energy barriers or harmonic KAM tori; **however, collective MD is sometimes very effective in highly constrained/dense molecular systems.** | Ergodicity is provable in principle; **you can invent clever non-local transition rules** (atom swaps, cluster flips) to make $\text{CPU time} / \text{equilibration}$ orders of magnitude faster. |

> **Best Practice (Slide 18):** Build **both MC and MD into the same simulation code** so you can **use MC (including non-local moves) to rapidly warm up and equilibrate the configuration, then switch to MD to compute real-time dynamics!**


In [48]:
ws.render_section('concept_step_size_optimizer')

### Interactive Explorer: Step Size $\Delta/\sigma$, Acceptance Ratio & Diffusion Efficiency

Vary the trial cube width $\Delta/\sigma$ and temperature scale to explore the liquid-Argon tradeoff from slides 15–16: compare the acceptance ratio $A(\Delta/\sigma)$ against the 100-step mean-squared displacement $\langle \Delta r^2(100) \rangle$, comparing the $A = 0.50$ rule of thumb with the $A \approx 0.20$ peak diffusion regime.

**Key Takeaway:** Neither $\Delta \to 0$ ($A \to 1$) nor $\Delta \to \infty$ ($A \to 0$) diffuses efficiently. While $A \approx 0.5$ ($\Delta/\sigma \approx 0.16$) is a safe rule of thumb, pushing $\Delta/\sigma \approx 0.36$ ($A \approx 0.20$) maximizes mean-squared displacement per CPU step in dense Lennard-Jones Argon.

In [49]:
ws.render_section('q15_37_step_size_extremes')

**Q37 · easy** · 💡

Why is an acceptance ratio of $99.9\%$ ($\text{a.r.} = 0.999$) in a continuous-space Metropolis simulation just as undesirable as an acceptance ratio of $0.1\%$ ($\text{a.r.} = 0.001$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Ask how large the trial displacement $\Delta$ must be to get $99.9\%$ acceptance in a dense fluid.

</div>
</details>

- [ ] **(A)** An acceptance ratio of $99.9\%$ means the trial step size $\Delta$ is tiny, so consecutive states are nearly identical and the random walk diffuses through phase space extremely slowly.
- [ ] **(B)** An acceptance ratio above $90\%$ violates detailed balance and produces a uniform distribution instead of a Boltzmann distribution.
- [ ] **(C)** High acceptance ratios cause a finite-step-size truncation error $\mathcal{O}(\Delta^2)$ in the equilibrium ensemble.
- [ ] **(D)** When $\text{a.r.} > 0.9$, the partition function $Z$ no longer cancels out.

In [50]:
ws.render_section('q15_38_liquid_argon_diffusion_vs_50pct')

**Q38 · medium** · 💡

According to the liquid Argon benchmarks on slides 15–16 of the lecture notes, how does a $20\%$ acceptance ratio ($\Delta/\sigma \approx 0.36$) compare to the conventional $50\%$ rule-of-thumb acceptance ratio ($\Delta/\sigma \approx 0.16$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare Fig. 1 (acceptance probability) and Fig. 3 ($\langle (\mathbf{r}(i) - \mathbf{r}(i+100))^2 \rangle$) on slide 15.

</div>
</details>

- [ ] **(A)** The $20\%$ acceptance ratio achieves higher 100-step mean-squared displacement (faster spatial diffusion) than the $50\%$ acceptance ratio because each accepted step covers more than twice the distance ($\Delta^2$ is $\sim 5\times$ larger).
- [ ] **(B)** The $20\%$ acceptance ratio gives zero diffusion because $80\%$ of moves are rejected.
- [ ] **(C)** The $50\%$ acceptance ratio always maximizes diffusion in every physical system.
- [ ] **(D)** The variance of the potential energy $\sigma_V$ diverges at $20\%$ acceptance.

In [51]:
ws.render_section('q15_39_local_energy_variance_sensitivity')

**Q39 · easy** · 💡

What do the liquid Argon benchmarks on slide 16 (Figs. 2 and 4) reveal about the sensitivity of the variance of a **local quantity** like total potential energy ($\sigma_V$) to the choice of step size $\Delta/\sigma$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the headline and plateau highlighted on slide 16.

</div>
</details>

- [ ] **(A)** The variance of the potential energy is relatively insensitive to step size across a wide acceptable range ($0.1 < \text{a.r.} < 0.9$), showing that MC is a robust method that does not require delicate fine-tuning of $\Delta$.
- [ ] **(B)** The potential energy variance is a sharp delta-function minimum at $\Delta/\sigma = 0.200$ and blows up if $\Delta$ changes by $5\%$.
- [ ] **(C)** The potential energy variance decreases monotonically as $\Delta/\sigma \to 0$.
- [ ] **(D)** Local energy variance can only be measured in Molecular Dynamics, not Monte Carlo.

In [52]:
ws.render_section('q15_40_no_timestep_error_in_mc')

**Q40 · medium** · 💡

In Molecular Dynamics, increasing the time step $\Delta t$ introduces systematic $\mathcal{O}(\Delta t^2)$ discretization errors into equilibrium averages. If you double the trial cube width $\Delta$ in an ergodic Metropolis Monte Carlo simulation (say from $\Delta = 0.15\sigma$ to $\Delta = 0.30\sigma$), what systematic discretization error is introduced into the converged equilibrium average $\langle A \rangle$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Does the detailed balance equation $\pi(s)P(s \to s') = \pi(s')P(s' \to s)$ make any small-$\Delta$ Taylor approximation?

</div>
</details>

- [ ] **(A)** Zero systematic discretization error: Metropolis MC is exact for any ergodic step size $\Delta > 0$ because the acceptance test enforces exact detailed balance for the true Boltzmann distribution $\pi(s)$.
- [ ] **(B)** The systematic error quadruples as $\mathcal{O}(\Delta^2)$, just like Verlet MD.
- [ ] **(C)** The systematic error doubles as $\mathcal{O}(\Delta)$.
- [ ] **(D)** Doubling $\Delta$ doubles the simulated temperature $T$.

In [53]:
ws.render_section('q15_41_md_vs_mc_capabilities')

**Q41 · medium** · 💡

Select **all** accurate statements comparing **Metropolis Monte Carlo (MC)** and **Molecular Dynamics (MD)** from slide 18:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check each statement against the MD vs. MC comparison on slide 18.

</div>
</details>

- [ ] **(A)** MD computes true physical real-time dynamics (e.g., velocity autocorrelation and vibrations), whereas MC generates stochastic Markov kinetics.
- [ ] **(B)** Classic MC is simpler than MD because it requires no forces (only energy differences), has no time-step discretization error, and directly samples the canonical ($NVT$) ensemble.
- [ ] **(C)** MC is more general for discrete degrees of freedom (such as Ising spin models, alloy species swaps, and grand canonical particle insertions) where continuous MD forces are undefined.
- [ ] **(D)** Combining both in the same code allows you to use fast, non-local MC moves to warm up (equilibrate) the structure and then switch to MD to measure real-time dynamics.

In [54]:
ws.render_section('q15_42_preview_force_bias_idea')

**Q42 · medium** · 💡

On slide 17 (*"Optimizing the moves"*), we preview how to go beyond blind uniform cube moves by expanding the potential around the current position $\mathbf{r}_0$:
$$V(\mathbf{r}) \approx V(\mathbf{r}_0) - \mathbf{F}(\mathbf{r}_0)\cdot(\mathbf{r} - \mathbf{r}_0)$$
and choosing a biased proposal distribution $T(s \to s') \propto \exp\big[\lambda \beta \mathbf{F}(\mathbf{r}_0)\cdot(\mathbf{r} - \mathbf{r}_0)\big]$. Why must we use the **generalized** Metropolis–Hastings acceptance formula $A(s \to s') = \min\!\left(1, \frac{T(s'\to s)\pi(s')}{T(s\to s')\pi(s)}\right)$ when we do this?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Ask whether $T(\mathbf{r}_0 \to \mathbf{r})$ equals $T(\mathbf{r} \to \mathbf{r}_0)$ when the proposal is pushed along the local force $\mathbf{F}$.

</div>
</details>

- [ ] **(A)** Because biasing the proposal in the direction of the downhill force makes $T(s \to s') \ne T(s' \to s)$ asymmetric, so the proposal ratio $T(s' \to s)/T(s \to s')$ must be included in $A(s \to s')$ to preserve exact detailed balance.
- [ ] **(B)** Because any use of forces in Monte Carlo makes the partition function $Z$ depend on the step number $n$.
- [ ] **(C)** Because $T(s \to s')$ is still symmetric ($T(s \to s') = T(s' \to s)$), so the $T$ factors cancel out anyway.
- [ ] **(D)** Because Force-Bias Monte Carlo does not use rejections.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()